# Imports

In [19]:
import pandas as pd
import sqlite3
import re
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase
from langchain_ollama import ChatOllama
from langchain.chains import create_sql_query_chain
from mistralai import Mistral
from sqlalchemy import text
from getpass import getpass

In [ ]:
# api_key= getpass("ysljhXZv3PXsbtrdAwUG0AqRCYq3SJFM")
client = Mistral(api_key="")
model = "mistral-small-2506"
chat_response = client.chat.complete(
    model= model,
    messages = [
        {
            "role": "user",
            "content": "What is the best French cheese?",
        },
    ]
)
print(chat_response.choices[0].message.content)
print(chat_response)

The "best" French cheese is highly subjective and depends on personal taste, but here are some of the most celebrated and iconic French cheeses that are often considered among the best:

### **1. Camembert de Normandie (AOP)**
   - A creamy, soft, and earthy cheese with a white mold rind.
   - Best enjoyed at room temperature for optimal flavor and texture.

### **2. Brie de Meaux (AOP)**
   - A rich, buttery, and slightly tangy cheese with a soft, creamy interior.
   - Often served with bread or fruit.

### **3. Roquefort (AOP)**
   - A bold, tangy blue cheese made from sheep’s milk.
   - One of the oldest and most famous French cheeses.

### **4. Comté (AOP)**
   - A nutty, firm, and slightly sweet cheese made from raw cow’s milk.
   - Great for melting in fondue or eating on its own.

### **5. Reblochon (AOP)**
   - A soft, creamy, and slightly funky cheese from the Alps.
   - Famous for its use in *Tartiflette* (a traditional dish).

### **6. Roquefort Société**
   - A premium vers

In [5]:
csv_path = "../resources/Crop_recommendation.csv"   # relative path from test.ipynb
db_path = "sqlite:///../resources/crops.db" 
df = pd.read_csv(csv_path)
print(df)

        N   P   K  temperature   humidity        ph    rainfall   label
0      90  42  43    20.879744  82.002744  6.502985  202.935536    rice
1      85  58  41    21.770462  80.319644  7.038096  226.655537    rice
2      60  55  44    23.004459  82.320763  7.840207  263.964248    rice
3      74  35  40    26.491096  80.158363  6.980401  242.864034    rice
4      78  42  42    20.130175  81.604873  7.628473  262.717340    rice
...   ...  ..  ..          ...        ...       ...         ...     ...
2195  107  34  32    26.774637  66.413269  6.780064  177.774507  coffee
2196   99  15  27    27.417112  56.636362  6.086922  127.924610  coffee
2197  118  33  30    24.131797  67.225123  6.362608  173.322839  coffee
2198  117  32  34    26.272418  52.127394  6.758793  127.175293  coffee
2199  104  18  30    23.603016  60.396475  6.779833  140.937041  coffee

[2200 rows x 8 columns]


In [6]:
# Save to SQLite
engine = create_engine(db_path)
df.to_sql("crops", con=engine, if_exists="replace", index=False)

2200

In [7]:
# 2. Setup SQLDatabase + LLM
# ------------------------------
db = SQLDatabase(engine)
llm = ChatOllama(model="gemma3:1b", temperature=0)  # Or any Ollama local model

chain = create_sql_query_chain(llm, db)

### Mistral

In [66]:
def extract_sql(answer: str) -> str:
    """
    Extract SQL code block or first SELECT statement from LLM output.
    """
    # If fenced code block
    match = re.search(r"```sql\n(.*?)```", answer, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # Otherwise, try to find the first SELECT
    match = re.search(r"(SELECT .*?;)", answer, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # fallback (return as-is, might fail)
    return answer.strip()


def ask_sql(query: str):
    prompt = f"""
    You are a data assistant.
    The table name is 'crops'. Schema: {df.dtypes.to_dict()}

    ONLY return a valid SQLite SQL query, no explanations.
    Query: "{query}"
    """
    response = client.chat.complete(
        model="mistral-small-2506",
        messages=[{"role": "user", "content": prompt}]
    )

    raw_output = response.choices[0].message.content.strip()
    sql_query = extract_sql(raw_output)

    # Run SQL
    with engine.connect() as conn:
        result = conn.execute(text(sql_query)).fetchall()
        explanation_prompt = f"""
    You are a helpful data assistant. 
    The user asked: "{query}"
    The SQL Query Generated: {sql_query}
    The SQL result is: {result}
    
    Please answer the question in natural language based on the result.
    """
    explanation = client.chat.complete(
        model="mistral-small",
        messages=[{"role": "user", "content": explanation_prompt}]
    )

    answer_sentence = explanation.choices[0].message.content.strip()
    return sql_query, result, answer_sentence

In [72]:
q="Give me best crops for each acidic , basic and neutral soils based on ph"
sql, res, ans = ask_sql(q)
sep="========================================================="
print(f"""\n\nQuestion:{q} \n{sep}\nSQL:{sql} \n{sep}\nResult:{res} \n{sep}\nAnswer:{ans}\n\n""")




Question:Give me best crops for each acidic , basic and neutral soils based on ph 
SQL:SELECT
    label AS crop,
    CASE
        WHEN ph < 7 THEN 'acidic'
        WHEN ph > 7 THEN 'basic'
        ELSE 'neutral'
    END AS soil_type,
    COUNT(*) AS frequency
FROM
    crops
GROUP BY
    label, soil_type
ORDER BY
    soil_type, frequency DESC; 
Result:[('apple', 'acidic', 100), ('banana', 'acidic', 100), ('coconut', 'acidic', 100), ('grapes', 'acidic', 100), ('kidneybeans', 'acidic', 100), ('maize', 'acidic', 100), ('mango', 'acidic', 100), ('muskmelon', 'acidic', 100), ('papaya', 'acidic', 100), ('watermelon', 'acidic', 100), ('pigeonpeas', 'acidic', 89), ('pomegranate', 'acidic', 84), ('mungbean', 'acidic', 78), ('rice', 'acidic', 73), ('coffee', 'acidic', 68), ('jute', 'acidic', 64), ('cotton', 'acidic', 55), ('lentil', 'acidic', 52), ('orange', 'acidic', 49), ('mothbeans', 'acidic', 48), ('blackgram', 'acidic', 36), ('chickpea', 'acidic', 36), ('blackgram', 'basic', 64), ('chickpe

In [56]:
questions=[
"What is the best crop for values around N=90, P=40, K=40, temperature around 25?",
"Which crop requires maximum rainfall?",
"Give ranges of temperature and humidity required for Rice.",
"List all unique crops suitable if rainfall > 200 and ph between 6 and 7."]
i=0
for q in questions:
    sql, res, ans = ask_sql(q)
    i=i+1
    print(f"""\n========================{i}=================================\nQuestion[{i}]:{q} \n\nSQL:{sql} \n\nResult:{res} \n\Answer:{ans}\n\n""")
    # print("SQL:", sql)
    # print("Result:", res)


========================1=================================
Question[1]:What is the best crop for values around N=90, P=40, K=40, temperature around 25? 

SQL:SELECT label, COUNT(*) as frequency
FROM crops
WHERE N BETWEEN 85 AND 95
  AND P BETWEEN 35 AND 45
  AND K BETWEEN 35 AND 45
  AND temperature BETWEEN 24 AND 26
GROUP BY label
ORDER BY frequency DESC
LIMIT 1; 

Result:[('jute', 6)] 
\Answer:Based on the data you've provided, it seems that the crop with the highest value score for the conditions N=90, P=40, K=40, and a temperature around 25 degrees is 'jute' with a score of 6. Please note that the score could be a ranking or a specific measure depending on your database schema.



========================2=================================
Question[2]:Which crop requires maximum rainfall? 

SQL:SELECT label, MAX(rainfall) as max_rainfall
FROM crops
GROUP BY label
ORDER BY max_rainfall DESC
LIMIT 1; 

Result:[('rice', 298.5601175)] 
\Answer:The crop that requires the most rainfall i

### Local ollama

In [26]:
def query_rag2(user_query: str):
    sql_query = chain.invoke({"question": user_query})
    print("\nGenerated SQL:", sql_query)

    result = db.run(sql_query)
    print("Result:", result)
    return result
def query_rag(user_query: str):
    sql_query = chain.invoke({"question": user_query})
    print("\nGenerated SQL:", sql_query)
    # Clean SQL (remove ``` blocks and "sql"/"sqlite" tags if present)
    sql_query = re.sub(r"```[\s\S]*?```", lambda m: m.group(0).replace("```", ""), sql_query)
    sql_query = sql_query.replace("sqlite", "").replace("sql", "").strip()
    
    print("\nGenerated SQL (cleaned):", sql_query)

    try:
        result = db.run(sql_query)
        print("Result:", result)
        return result
    except Exception as e:
        print("Execution error:", e)
        return None

In [27]:
# ------------------------------
# 4. Example queries
# ------------------------------
query_rag("What is the best crop for N=90, P=40, K=40, temperature around 25?")
query_rag("Which crop requires maximum rainfall?")
query_rag("Give average temperature and humidity required for Rice.")
query_rag("List all crops suitable if rainfall > 200 and ph between 6 and 7.")


Generated SQL: ```sqlite
SELECT "N", "P", "K", "temperature" FROM crops LIMIT 5
```

Generated SQL (cleaned): SELECT "N", "P", "K", "temperature" FROM crops LIMIT 5
Result: [(90, 42, 43, 20.87974371), (85, 58, 41, 21.77046169), (60, 55, 44, 23.00445915), (74, 35, 40, 26.49109635), (78, 42, 42, 20.13017482)]

Generated SQL: SELECT "K" FROM crops ORDER BY "rainfall" DESC LIMIT 5

Generated SQL (cleaned): SELECT "K" FROM crops ORDER BY "rainfall" DESC LIMIT 5
Result: [(40,), (44,), (37,), (37,), (40,)]

Generated SQL: SELECT "temperature", "humidity" FROM crops WHERE "label" = 'rice'

Generated SQL (cleaned): SELECT "temperature", "humidity" FROM crops WHERE "label" = 'rice'
Result: [(20.87974371, 82.00274423), (21.77046169, 80.31964408), (23.00445915, 82.3207629), (26.49109635, 80.15836264), (20.13017482, 81.60487287), (23.05804872, 83.37011772), (22.70883798, 82.63941394), (20.27774362, 82.89408619), (24.51588066, 83.53521629999999), (23.22397386, 83.03322691), (26.52723513, 81.4175384

'[(90, 42), (74, 35), (89, 54), (68, 58), (94, 50), (85, 38), (91, 35), (67, 59), (83, 41), (97, 59), (60, 49), (92, 35), (85, 37), (88, 54), (62, 42), (83, 60), (82, 40), (76, 60), (75, 38), (99, 55), (93, 58), (70, 36), (86, 59), (91, 56), (61, 52), (79, 42), (97, 36), (61, 53), (66, 60), (91, 50), (81, 45), (60, 51), (93, 47), (61, 68), (39, 65), (70, 68), (38, 68), (44, 64), (34, 62), (47, 46), (32, 68), (50, 59), (65, 62), (50, 47), (36, 54), (37, 52), (33, 47), (40, 49), (42, 53), (44, 47), (52, 51), (35, 68), (39, 69), (49, 61), (53, 55), (56, 65), (58, 55), (35, 67), (39, 64), (18, 30), (0, 19), (16, 14), (23, 6), (24, 14), (29, 8), (37, 10), (15, 28), (3, 23), (2, 30), (3, 9), (27, 8)]'